In [1]:
import wandb


c:\Users\Csaba\anaconda3\envs\myenv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\Csaba\anaconda3\envs\myenv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` 

In [2]:
wandb.login()

wandb: Currently logged in as: csczuth (csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import pandas as pd
import wandb
import json

api = wandb.Api()
entity, project = "csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem", "federated-learning"
runs = api.runs(entity + "/" + project)

summary_list, config_list, accuracies, name_list, sys_usage_dfs, strategy, clients, rounds = [], [], [], [], [], [], [], []
for run in runs:
    # .summary contains the output keys/values
    #  for metrics such as accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    print(run)
    config_list.append({})
    tmp_df = run.history(stream="events")
    tmp_df["run_id"] = run.id
    sys_usage_dfs.append(tmp_df)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    s = json.loads(run.config)
    rounds_tmp, strategy_tmp, clients_num_tmp = s.get("number_of_rounds", {}).get("value"), s.get("strategy", {}).get("value"), s.get("min_clients", {}).get("value")
    rounds.append(rounds_tmp)
    strategy.append(strategy_tmp)
    clients.append(clients_num_tmp)
    try:
        #config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
        tmp_acc = []
        # if run.state == "finished":
        for i, row in run.history().iterrows():
            tmp_acc.append(row["accuracy"])
        accuracies.append(tmp_acc)
    except Exception as e:
        #config_list.append({})
        print('erroor in accuracy extraction for run ', run.id)
        accuracies.append([])

    # .name is the human-readable name of the run.
    name_list.append(run.id)
    

runs_df = pd.DataFrame(
    {#"summary": summary_list, "config": config_list, 
     "accuracies": accuracies, "name": name_list, "rounds": rounds, "strategy": strategy, "clients_num": clients}
)

runs_df.to_csv("project.csv")
# for s in sys_usage_dfs:
#     print(s)

<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/mtblu6fm (crashed)>
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/2rs2p3o8 (finished)>
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/jrtv3cqw (finished)>
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/583zq665 (finished)>
erroor in accuracy extraction for run  583zq665
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/hi88bhix (finished)>
erroor in accuracy extraction for run  hi88bhix
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/2pf362ki (finished)>
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/v4uf1jk4 (finished)>
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/laapbv3g (finished)>
<Run csczuth-budapesti-m-szaki-s-gazdas-gtudom-nyi-egyetem/federated-learning/axkiqeth (crashed)>
<Run csczuth-bu

In [4]:
runs_df['max_accuracy'] = runs_df.apply(lambda row: max(row["accuracies"]) if len(row["accuracies"]) > 0 else 0, axis=1)

In [5]:
fedavgs = runs_df[(runs_df['strategy'] == 'FedAvg') & (runs_df['rounds'] <= 100) & (runs_df['max_accuracy'] < 0.66)]
#fedavgs
fedavgs.groupby('clients_num')['max_accuracy'].max().reset_index().to_csv("scalability.csv", index=False)

In [6]:
fedadams = runs_df[(runs_df['strategy'] == 'FedAdam') | (runs_df['strategy'] == 'FedAdamDefault')]

In [7]:
fedadams

,accuracies,name,rounds,strategy,clients_num,max_accuracy
45,"[0.28413793103448276, 0.38068965517241377, 0.3...",2fuba1sy,10.0,FedAdamDefault,2.0,0.431724
137,[],8gerv1dy,100.0,FedAdam,2.0,0.000000
138,"[0.30896551724137933, 0.3020689655172414, 0.26...",592yyio6,100.0,FedAdam,2.0,0.308966
139,"[0.19724137931034483, 0.3255172413793103, 0.30...",0pv3cfgu,100.0,FedAdam,2.0,0.337931
158,"[0.31336405529953915, 0.29493087557603687, 0.2...",nb1t0l88,10.0,FedAdamDefault,32.0,0.373272
159,[0.2798165137614679],mu7mjmeh,10.0,FedAdamDefault,32.0,0.279817
160,"[0.2672811059907834, 0.2626728110599078, 0.299...",8z0yadky,10.0,FedAdamDefault,32.0,0.377880
161,"[0.2488479262672811, 0.30414746543778803, 0.27...",fo2z8qg0,10.0,FedAdamDefault,32.0,0.341014
162,"[0.2857142857142857, 0.25806451612903225, 0.23...",pxmjb3lk,400.0,FedAdamDefault,32.0,0.428571
165,"[0.26146788990825687, 0.2857142857142857, 0.26...",i0a4jqp5,100.0,FedAdamDefault,32.0,0.506912


In [19]:
strats = runs_df[(runs_df['rounds'] >= 100) & (runs_df['clients_num'] > 4)]
strats = strats[~strats['strategy'].str.contains('Default')]

In [21]:
strats.groupby("strategy").agg(max_accuracy=("max_accuracy", "max"),client_nums=("clients_num", "first")).reset_index()
#.to_csv("strategy_comparison.csv", index=False)
#.to_csv("scalability.csv", index=False)

,strategy,max_accuracy,client_nums
0,FedAdagrad,0.506912,32.0
1,FedAdam,0.493088,32.0
2,FedAvg,0.824885,32.0
3,FedAvgM,0.697248,32.0
4,FedCM,0.555046,32.0
5,FedProx,0.788018,32.0
6,FedTrimmedAvg,0.587156,32.0
7,FedYogi,0.511521,32.0
